[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/experiments/building_your_own_eval_harness.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://docs.arize.com/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-1px8dcmlf-fmThhDFD_V_48oU7ALan4Q">社区</a>
    </p>
</center>


# 为什么公开基准会误导你：构建自己的评估框架

*完整示例：邮件文本提取服务。*

## 榜首模型可能输在你的任务上

新模型登顶 MMLU、GPQA 或最新排行榜时，很容易认定它也最适合*你的*应用。事实通常并非如此，至少理由并不是你想的那样。公开基准使用**通用数据**和**通用指标**衡量**通用能力**：对数千个与生产流量截然不同的问题，以选择题准确率衡量广泛的学术知识，再取平均值。

你的任务更具体、更狭窄。你并不是让模型回答知识问答，而是要求它每次都从客户邮件中准确提取五个字段，并返回下游代码可以解析的结构。MMLU 高两分的模型，在*你的*任务上可能低二十分，因为：

- **数据不同。**你的输入是客户邮件，而非试题。
- **指标不同。**你关心 `due_date` 是否完全正确，而非文字是否显得聪明。
- **失败模式不同。**对你而言，自信且貌似可信的错误答案，比“不知道”更糟；这与知识基准的奖励方向相反。

唯一能预测模型在你的任务上表现的基准，是**根据你的任务构建的基准**。

## 什么样的基准值得信任

信任*任何*数字之前，不论来自公开基准还是自建基准，都应检查其是否具有以下四个属性：

1. **有代表性的数据。**示例取自真实输入，覆盖实际遇到的情况，包括杂乱和模糊的情况，而非干净的玩具样本。
2. **衡量你关心内容的指标。**只有当输出对*你的用途*更好时分数才上升，否则保持不变。错误指标可能把更差的模型排在第一，下面会具体演示。
3. **足够多的示例以保持稳定。**两个示例无法区分两个模型，分数中的信号必须多于噪声。
4. **可复现且公平。**所有模型使用*相同*数据集、指标和提示词接受评判，让分差反映模型差异，而非设置差异。

Phoenix **实验**正是这样的框架：固定的**数据集**、可以改变的**任务**及一个或多个**评估器**。保持数据集和评估器不变，只替换模型，便可从结构上保证比较公平。

**本 Notebook 将带你：**

- 构建包含邮件及正确提取结果的小型**领域数据集**，作为自己的基准
- 定义具有固定结构与提示词、仅以模型为参数的**提取任务**
- 定义**两个评估器**，分别计算字符串相似度和字段准确率，观察它们如何给模型不同排名
- 在 `gpt-5.4-mini` 和旗舰 `gpt-5.5` 上运行**相同框架**，并在 Phoenix 中公平比较

✅ 运行本 Notebook 需要免费的 [Phoenix Cloud 账号](https://app.arize.com/auth/phoenix/login) 和 OpenAI API 密钥。

# 配置密钥与依赖

In [ ]:
%pip install arize-phoenix arize-phoenix-otel arize-phoenix-client openai openinference-instrumentation-openai jarowinkler pandas pydantic nest_asyncio

In [ ]:
import os
from getpass import getpass

if not (phoenix_endpoint := os.getenv("PHOENIX_COLLECTOR_ENDPOINT")):
    phoenix_endpoint = getpass("🔑 请输入 Phoenix 收集器端点：")
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = phoenix_endpoint

if not (phoenix_api_key := os.getenv("PHOENIX_API_KEY")):
    phoenix_api_key = getpass("🔑 请输入 Phoenix API 密钥：")
os.environ["PHOENIX_API_KEY"] = phoenix_api_key

if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥：")
os.environ["OPENAI_API_KEY"] = openai_api_key

# 配置追踪

注册追踪器提供程序并自动为 OpenAI 插桩，使每次提取调用都作为 span 出现在 Phoenix 中。启用追踪后，实验结果可链接回生成它们的具体调用。

In [ ]:
import nest_asyncio
from phoenix.otel import register

nest_asyncio.apply()  # 允许在 Notebook 中运行异步实验代码

# auto_instrument=True 激活已安装的 OpenInference 插桩器，此处为 OpenAI，
# 因此无需单独调用 OpenAIInstrumentor().instrument(...)。
register(project_name="email-extraction-eval-harness", auto_instrument=True)

In [ ]:
from phoenix.client import AsyncClient

px_client = AsyncClient()

# 构建领域数据集

这是公开基准无法替你完成的部分。我们按服务实际收到的内容，手动标注几封邮件，包括会议请求、发票、客服升级和销售回复，每封都配有所需的**精确**结构化输出。

生产环境中应从真实流量构建：从 Phoenix 导出追踪、采样并标注。这里内嵌一个小数据集，使 Notebook 自包含。每个示例都有 `email` 输入和含五个字段的 `expected` 提取结果：

- `sender`：发件人，自由文本
- `category`：固定集合中的一类，分类字段
- `summary`：一句话概要，自由文本
- `action_required`：是否需要人工采取行动，布尔值
- `due_date`：ISO 日期或 `"none"`，近似分类字段

自由文本与分类字段的混合很重要：它使两种合理指标稍后可能产生*分歧*。

> **这是演示框架。**八个示例足以说明流程，但不足以得出稳定结论，参见上述第 3 个属性。生产框架需要从真实流量抽取更大、有代表性的样本，排名才值得信任。

In [ ]:
EMAILS = [
    {
        "email": "From: Dana Whitfield <dana@acme.co>\nSubject: Kickoff for Q3 redesign\n\nHi team, can we lock in the kickoff for the Q3 site redesign? I'm proposing Thursday July 10 at 2pm PT. Please confirm by end of week so I can send invites.",
        "expected": {
            "sender": "dana@acme.co",
            "category": "meeting",
            "summary": "Proposes a Q3 redesign kickoff meeting on July 10 at 2pm PT.",
            "action_required": True,
            "due_date": "2025-07-10",
        },
    },
    {
        "email": "From: billing@cloudhost.com\nSubject: Invoice #88231 due\n\nYour CloudHost invoice #88231 for $4,200.00 is due on 2025-06-30. Please remit payment to avoid a service interruption.",
        "expected": {
            "sender": "billing@cloudhost.com",
            "category": "invoice",
            "summary": "CloudHost invoice #88231 for $4,200 is due June 30.",
            "action_required": True,
            "due_date": "2025-06-30",
        },
    },
    {
        "email": "From: Marcus Lee <m.lee@biotechlabs.org>\nSubject: URGENT: dashboard down\n\nOur production dashboard has been returning 500 errors since this morning and our analysts are blocked. We need this resolved today. Ticket #4471 already open.",
        "expected": {
            "sender": "m.lee@biotechlabs.org",
            "category": "support_request",
            "summary": "Production dashboard is down with 500 errors; needs resolution today.",
            "action_required": True,
            "due_date": "none",
        },
    },
    {
        "email": "From: Priya Nair <priya@growthpartners.io>\nSubject: Re: enterprise pricing\n\nThanks for the deck. Leadership is interested but wants to see SOC 2 docs before we go further. No rush on timing.",
        "expected": {
            "sender": "priya@growthpartners.io",
            "category": "sales",
            "summary": "Interested in enterprise plan but needs SOC 2 docs before proceeding.",
            "action_required": True,
            "due_date": "none",
        },
    },
    {
        "email": "From: HR <people@acme.co>\nSubject: Office closed Monday\n\nReminder: the office will be closed Monday for the holiday. No action needed, just a heads up. Enjoy the long weekend!",
        "expected": {
            "sender": "people@acme.co",
            "category": "internal_update",
            "summary": "Office is closed Monday for the holiday; informational only.",
            "action_required": False,
            "due_date": "none",
        },
    },
    {
        "email": "From: Tom Briggs <tom@vendorworks.com>\nSubject: Contract renewal by Aug 1\n\nOur MSA is up for renewal. To keep continuity, please sign and return the renewal addendum no later than August 1, 2025. Reach out with redlines.",
        "expected": {
            "sender": "tom@vendorworks.com",
            "category": "sales",
            "summary": "MSA renewal addendum must be signed and returned by August 1.",
            "action_required": True,
            "due_date": "2025-08-01",
        },
    },
    {
        "email": "From: alerts@statuspage.io\nSubject: All systems operational\n\nThis is your weekly status digest. All monitored services were operational over the past 7 days with 100% uptime. No incidents reported.",
        "expected": {
            "sender": "alerts@statuspage.io",
            "category": "internal_update",
            "summary": "Weekly status digest: all services operational, 100% uptime, no incidents.",
            "action_required": False,
            "due_date": "none",
        },
    },
    {
        "email": "From: Sofia Reyes <sofia@designco.studio>\nSubject: Need feedback on mockups\n\nDropped v2 of the mockups in the shared drive. Could you review and send comments before our sync on June 25? Want to finalize before the build phase.",
        "expected": {
            "sender": "sofia@designco.studio",
            "category": "support_request",
            "summary": "Requests review of v2 mockups before the June 25 sync.",
            "action_required": True,
            "due_date": "2025-06-25",
        },
    },
]

In [ ]:
from datetime import datetime, timezone

import pandas as pd
from phoenix.client.utils.config import get_base_url

# 将各示例的预期提取结果展开为顶层列，使数据集的
# 输出就是评估器用于比较的提取字典，而非嵌套封装。
rows = [{"email": e["email"], **e["expected"]} for e in EMAILS]
df = pd.DataFrame(rows)
OUTPUT_KEYS = ["sender", "category", "summary", "action_required", "due_date"]

dataset = await px_client.datasets.create_dataset(
    name=f"email-extraction-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}",
    dataframe=df,
    input_keys=["email"],
    output_keys=OUTPUT_KEYS,
)

# 打印数据集直达链接，便于查看刚上传的示例。
base_url = str(get_base_url()).rstrip("/")
print(f"已上传 {len(df)} 个示例到数据集 '{dataset.name}'")
print(f"在 Phoenix 中查看数据集： {base_url}/datasets/{dataset.id}/examples")

# 定义提取任务

任务保持*几乎*不变：相同结构、相同提示词、相同解析方式，**只改变模型**。使用 OpenAI 结构化输出，强制所有模型返回完全相同的形状，让比较关注提取质量，而非碰巧生成正确格式。

`make_task(model)` 返回绑定到一个模型的任务函数。实验对每个示例调用一次；传入的 `input` 是数据集示例的输入字典（`{"email": ...}`）。

In [ ]:
from typing import Literal

from openai import AsyncOpenAI
from pydantic import BaseModel

openai_client = AsyncOpenAI()


class EmailExtraction(BaseModel):
    sender: str
    category: Literal["meeting", "invoice", "support_request", "sales", "internal_update"]
    summary: str
    action_required: bool
    due_date: str  # ISO 日期（YYYY-MM-DD）或字面字符串 "none"


PROMPT = (
    "从下面的邮件中提取结构化字段。"
    "sender 必须是发件人的电子邮箱。"
    "category 必须为 meeting、invoice、support_request、sales、internal_update 之一。"
    "若邮件明确截止日期，due_date 必须是 ISO 日期（YYYY-MM-DD），"
    '否则为字面字符串 "none"。'
    "若邮件要求收件人采取行动，则 action_required 为 true。\n\n邮件：\n{email}"
)


def make_task(model: str):
    async def task(input) -> dict:
        response = await openai_client.beta.chat.completions.parse(
            model=model,
            messages=[{"role": "user", "content": PROMPT.format(email=input["email"])}],
            response_format=EmailExtraction,
            # 保留模型默认温度；公平比较只改变
            # 模型，且最新模型只接受默认采样设置。
        )
        return response.choices[0].message.parsed.model_dump()

    return task

In [ ]:
# 完整实验前，先用一个示例检查任务。
await make_task("gpt-5.4-mini")(dataset.examples[0]["input"])

# 选择衡量你关心内容的指标

基准往往在这里悄悄产生误导。我们用两种方式给**相同**输出评分：

- **`jaro_winkler`**：自由文本 `summary` 字段的字符串相似度。它成本低、容忍度高，是人们首先想到的“看起来差不多”指标。对措辞不同但可能同样正确的摘要而言，这是合适的工具。
- **`field_accuracy`**：**业务操作字段**（`sender`、`category`、`action_required`、`due_date`）中*精确*匹配的比例，忽略大小写。下游代码依赖这些字段：“接近”的截止日期仍是错误日期。这里特意排除 `summary`，因为精确匹配会惩罚无害的改写。

两种指标衡量不同内容，因此*可能*给模型不同排名：一个模型可能摘要更好（`jaro_winkler` 更高），却错分更多邮件（`field_accuracy` 更低），反之亦然。出现分歧时，应由与下游需求相关的指标决定，本例是 `field_accuracy`。下面先用确定性示例说明，再观察两个真实模型的表现。

In [ ]:
import jarowinkler

# jaro_winkler 只评估自由文本摘要；field_accuracy 只判断
# 下游代码实际依赖的业务字段。二者不重叠，
# 因此可以独立变化。
OPERATIONAL_FIELDS = ["sender", "category", "action_required", "due_date"]


def jaro_winkler(output, expected) -> float:
    """对自由文本摘要使用宽松的字符串相似度，措辞不同但正确的摘要仍可获得高分。"""
    return jarowinkler.jarowinkler_similarity(
        str(output.get("summary", "")),
        str(expected["summary"]),
    )


def field_accuracy(output, expected) -> float:
    """业务操作字段中精确匹配的比例，忽略大小写。"""
    matches = sum(
        1
        for k in OPERATIONAL_FIELDS
        if str(output.get(k)).strip().lower() == str(expected[k]).strip().lower()
    )
    return matches / len(OPERATIONAL_FIELDS)


EVALUATORS = [jaro_winkler, field_accuracy]

运行模型前，先用确定性示例展示分歧。同一封发票邮件有两个候选提取结果：一个业务字段正确但改写了 `summary`，另一个看起来几乎相同，却将 `due_date` 写错了一天。

In [ ]:
expected = {
    "sender": "billing@cloudhost.com",
    "category": "invoice",
    "action_required": True,
    "due_date": "2025-06-30",
    "summary": "CloudHost invoice #88231 for $4,200 is due June 30.",
}
# A：所有业务字段正确，但摘要完全改写，不影响含义。
candidate_a = {
    **expected,
    "summary": "Please arrange payment for the recent cloud hosting charges before the close of the current billing period.",
}
# B：摘要完全相同，但 due_date 错了一天。
candidate_b = {**expected, "due_date": "2025-07-01"}

for name, cand in [
    ("A（字段正确，摘要改写）", candidate_a),
    ("B（摘要相同，due_date 错一天）", candidate_b),
]:
    print(
        f"{name}: jaro_winkler={jaro_winkler(cand, expected):.3f}  "
        f"field_accuracy={field_accuracy(cand, expected):.3f}"
    )

# field_accuracy 偏好 A，因所有业务字段正确；jaro_winkler 偏好 B，
# 因其看起来几乎相同。但 B 的一天日期偏差恰好会破坏
# 下游代码。同样的输出得到相反排名，而严格指标是正确的。

# 在不同模型上运行相同框架

相同数据集、评估器和提示词，只修改 `model` 参数。这让比较公平，而非仅凭个例判断。

In [ ]:
experiment_full = await px_client.experiments.run_experiment(
    dataset=dataset,
    task=make_task("gpt-5.5"),
    evaluators=EVALUATORS,
    experiment_name="gpt-5.5",
)

In [ ]:
experiment_mini = await px_client.experiments.run_experiment(
    dataset=dataset,
    task=make_task("gpt-5.4-mini"),
    evaluators=EVALUATORS,
    experiment_name="gpt-5.4-mini",
)

# 公平比较

Phoenix 会打印每次实验的摘要，UI 支持逐示例比较两次运行。这里也按指标汇总评分，使分歧清晰可见。

In [ ]:
from collections import defaultdict


def average_scores(experiment) -> dict:
    sums, counts = defaultdict(float), defaultdict(int)
    for run in experiment["evaluation_runs"]:
        result = run.result
        if result and result.get("score") is not None:
            sums[run.name] += result["score"]
            counts[run.name] += 1
    return {name: sums[name] / counts[name] for name in sums}


summary = pd.DataFrame(
    {"gpt-5.5": average_scores(experiment_full), "gpt-5.4-mini": average_scores(experiment_mini)}
)
print(summary)

在 Phoenix 中打开实验比较，查看平均值背后的逐示例详情。只有八个示例，这次运行未必能清晰区分两个模型，这正说明为什么要*观察*而非假定。无论结果如何，核心结论都成立：

- 如果两个指标**给模型不同排名**，照搬公开排行榜“选高分模型”的直觉可能导致错误选择。哪个模型“更好”取决于指标，而与下游需求相关的指标应优先，本例为 `field_accuracy`。
- 如果它们**一致**，你就获得了以*自己的*数据和指标为依据的证据，而非借用一个从未见过你任务的基准数字。

无论哪种情况，因为亲自构建了框架，你都能有依据地信任结果。

# 在 Phoenix 中阅读结果

打开数据集的 **Experiments** 选项卡，并排查看运行。每个实验占一行，每个评估器都有独立的**分数列**，因此 `field_accuracy` 与 `jaro_winkler` 并列显示；还包含平均**延迟**、**成本**和**错误率**等业务指标。这些对真实模型选择很重要，却不会出现在公开排行榜上。

![在 Phoenix 中比较 gpt-5.5 与 gpt-5.4-mini 实验](https://storage.googleapis.com/arize-phoenix-assets/assets/images/email-eval-harness-experiments.png)

两个指标*可能*给模型不同排名：摘要更好的模型（`jaro_winkler` 更高）仍可能错分更多邮件（`field_accuracy` 更低），反之亦然。出现分歧时，应按下游需求相关的指标（`field_accuracy`）排序，而非按更宽松的指标。点击任意行进入示例级视图，查看输入邮件、模型提取结果和各评估器对此示例的评分。你可以在那里*看清为什么*某个模型胜出，比如另一个模型遗漏了 `due_date` 或提取了过多发件人内容，而非仅信任聚合值。

![Phoenix 示例级比较视图，展示输入、参考结果、模型输出与各示例评分](https://storage.googleapis.com/arize-phoenix-assets/assets/images/email-eval-harness-example-level.png)

# 后续方向

框架可以复用。以下每一步都保持数据集和评估器不变，每次只改变一个因素，确保比较公平：

- **迭代提示词。**修改 `PROMPT` 而非 `model`，找出提取最可靠的措辞。
- **增加模型与提供商。**在 `make_task` 中替换其他模型名或提供商客户端，然后重新运行。排行榜名次往往经不起实际任务检验。
- **扩大数据集。**从八个内嵌示例扩展为从真实流量导出的代表性样本，使评分足够稳定、值得信任，即第 3 个属性。
- **形成持续基准。**每次模型升级或提示词变化都重新运行相同框架，在回归进入生产前发现它。

# 要点

公开基准告诉你的是模型在*别人*的任务、按*别人*的指标表现如何。这个数字很少能直接迁移到生产。

解决方法是建立小而可信的自有框架：

1. **代表性数据**：从真实输入构建数据集。
2. **衡量实际关注内容的指标**：留意便利指标（字符串相似度）与实际指标（字段准确率）的分歧。
3. **足够多的示例**：保证评分稳定。
4. **公平、可重复的设置**：相同数据集、评估器和提示词，只改变模型。每次运行的采样会带来少量噪声，但固定这些因素后，分差仍可归因于模型；重点在此，而非逐比特完全复现。

Phoenix 实验提供这四项能力。框架建好后，比较模型、提示词或提供商，只需替换一个参数，再读取一个有据可依的分数。